# 18 · Calibration Workflows

Finstack ships a calibration helper that ingests market quotes, solves curves, and returns enriched `MarketContext` objects. This notebook wires the test quote set into `SimpleCalibration`.

### Learning Objectives
- Load reusable quote sets (rates, forwards, credit, vols)
- Configure `SimpleCalibration` with solver options
- Inspect calibration reports and derived market data

In [7]:
from datetime import date, timedelta
from pathlib import Path
import sys

from finstack.core.dates.schedule import Frequency
from finstack.core.market_data import MarketContext
from finstack.valuations import calibration as cal

# Find project root (go up from current notebook location)
current_dir = Path.cwd()
project_root = current_dir
while not (project_root / "finstack-py").exists() and project_root != project_root.parent:
    project_root = project_root.parent

helper_dir = project_root / "finstack-py" / "examples" / "scripts" / "valuations" / "calibration"
if str(helper_dir) not in sys.path:
    sys.path.append(str(helper_dir))

# Always define base_date
base_date = date(2024, 1, 2)

try:
    import calibration_capabilities as calib_helpers
    market_quotes, forward_inputs, credit_inputs, swaption_specs = calib_helpers.build_market_quotes(base_date)
except ImportError as e:
    print(f"Warning: Could not import calibration helpers: {e}")
    print("Skipping helper-based calibration...")

## 1. Discount Curve Calibration
We start by calibrating a USD-OIS discount curve directly with `DiscountCurveCalibrator`. The helper returns market quotes, so we recreate the short deposit and swap points inline before bootstrapping.

In [8]:
discount_quotes = [
    cal.RatesQuote.deposit(base_date + timedelta(days=30), 0.0450, "ACT/360"),
    cal.RatesQuote.deposit(base_date + timedelta(days=90), 0.0465, "ACT/360"),
    cal.RatesQuote.swap(
        base_date + timedelta(days=365),
        0.0475,
        Frequency.ANNUAL,
        Frequency.QUARTERLY,
        "30/360",
        "ACT/360",
        "USD-SOFR",
    ),
    cal.RatesQuote.swap(
        base_date + timedelta(days=365 * 3),
        0.0485,
        Frequency.ANNUAL,
        Frequency.QUARTERLY,
        "30/360",
        "ACT/360",
        "USD-SOFR",
    ),
]

market = MarketContext()
discount_calibrator = cal.DiscountCurveCalibrator("USD-OIS", base_date, "USD")
discount_calibrator = discount_calibrator.with_config(
    cal.CalibrationConfig.multi_curve()
    .with_solver_kind(cal.SolverKind.HYBRID)
    .with_max_iterations(100)
)
discount_curve, discount_report = discount_calibrator.calibrate(discount_quotes)
market.insert_discount(discount_curve)
print("Discount curve calibrated:", discount_curve.id)
print("Report success:", discount_report.success)
print("Number of knots:", len(discount_curve.points))

Discount curve calibrated: USD-OIS
Report success: True
Number of knots: 5


# 2. Forward, Credit, and Volatility Calibrations
# With the discount curve loaded into a `MarketContext`, reuse the helper utilities
# to calibrate tenor-specific forwards, hazard curves, and populate swaption vol
# surfaces. These helpers are thin wrappers around the public calibrator types.

In [10]:
try:
    forward_reports = calib_helpers.calibrate_forward_curves(market, base_date, forward_inputs)
    credit_reports = calib_helpers.calibrate_credit_index_structures(market, base_date, credit_inputs)
    fallback_surface = calib_helpers.ensure_swaption_surface(market, base_date, swaption_specs)
    print("Forward curves calibrated:", list(forward_reports.keys()))
    print("Credit objects calibrated:", list(credit_reports.keys()))
    print("Swaption surface populated?", bool(fallback_surface))
except NameError as e:
    print(f"Skipping: calibration helpers not available ({e})")

Forward curves calibrated: ['USD-SOFR-3M-FWD', 'USD-SOFR-6M-FWD']
Credit objects calibrated: ['CDX.NA.IG-senior', 'CDX-IG-BC', 'CDX.NA.IG']
Swaption surface populated? True


# 3. Summarize Results
# Display calibrated curve IDs, forward/hazard reports, and sample vol data via the
# helper `summarize_context`. Inspect the aggregated context to confirm curve counts
# and metadata.

In [14]:
try:
    calib_helpers.summarize_context(market, forward_reports, credit_reports)
    if fallback_surface:
        print("Swaption surface bootstrapped from sample ATM grid")
except NameError:
    print("Skipping: calibration helpers not available")


Curves: {'Discount': 1, 'BaseCorrelation': 1, 'Forward': 2, 'Hazard': 1}
Vol surfaces: 1
USD-SOFR-3M-FWD calibrated (iterations=1, max residual=0.002539)
USD-SOFR-6M-FWD calibrated (iterations=2, max residual=0.006668)
CDX.NA.IG-senior available
CDX-IG-BC available
  note: points: [(0.03, 0.1), (0.07, 0.14), (0.1, 0.17), (0.15, 0.21), (0.3, 0.25)]
CDX.NA.IG available
  note: credit index registered
USD-OIS df(5y): 0.788374
USD-SOFR-3M-FWD rate(2y): 0.049473
USD-SOFR-6M-FWD rate(2y): 0.0505
ACME hazard curve missing
US-CPI-U inflation curve missing
CDX base correlation 7%:  0.14
ACME-VOL surface missing
Swaption vol 2.0y / strike 6.0: 0.231
Swaption surface bootstrapped from sample ATM grid


## Summary
- `calibration_capabilities.build_market_quotes` bundles representative USD quotes across rates, credit, and vol.
- `SimpleCalibration` plus solver config calibrates discount/forward curves, credit spreads, and fills volatility surfaces.
- Helper utilities summarize the resulting `MarketContext`, ready for downstream pricing/risk.